# ZemaVoice Stage 11B — OmniVoice Installation Fix + Female Amharic Test

Stage 11 failed because `pip install omnivoice` could not locate the package. This notebook uses the **GitHub source installation documented by k2-fsa** and the documented `ref_audio` API. **Experimental, not verified on your Colab runtime yet.** Install may fail on some Colab Python/CUDA versions. Use a T4 GPU (Runtime → Change runtime type). You confirmed you have consent to use the female voice recording.

**Licenses:** The base `k2-fsa/OmniVoice` weights are noncommercial; check the Amharic model's actual terms before use beyond personal evaluation. Do not distribute the reference recording or cloned voice without the speaker's consent.


In [ ]:
import sys, torch
print('Python',sys.version)
print('Torch',torch.__version__,'CUDA',torch.cuda.is_available())
assert torch.cuda.is_available(),'Select Runtime → Change runtime type → T4 GPU, then rerun.'


In [ ]:
import subprocess,sys
cmd=[sys.executable,'-m','pip','install','-q','git+https://github.com/k2-fsa/OmniVoice.git','soundfile']
p=subprocess.run(cmd,capture_output=True,text=True)
print('Installation return code:',p.returncode)
if p.returncode:
 print(p.stderr[-6500:]);raise RuntimeError('GitHub installation failed. Send the output above to ChatGPT.')
print('Installed OmniVoice from GitHub. If imports fail, restart the Colab session and rerun from the top.')


In [ ]:
import soundfile as sf, numpy as np, torch
from omnivoice import OmniVoice
from IPython.display import Audio,display
from google.colab import files
print('Import succeeded.')


In [ ]:
# Model repository listed on the Amharic model card; fallback to its visible repository.
model=None
for repo in ['african-low-resource/omnivoice-amharic','gheero-Leyu/amharic-omnivoice-tts']:
 try:
  print('Trying:',repo)
  model=OmniVoice.from_pretrained(repo,device_map='cuda:0',dtype=torch.float16)
  print('Loaded:',repo)
  break
 except Exception as e:
  print(type(e).__name__,str(e)[:1000])
assert model is not None,'Model loading failed. Share the errors above; model repository compatibility may differ.'


In [ ]:
uploaded=files.upload()
assert uploaded,'Upload zemavoice_stage13_female_reference.wav from your Mac.'
ref=next(iter(uploaded))
print('Reference:',ref)
display(Audio(filename=ref))


In [ ]:
text='ሰላም፣ ወደ ዜማ ቮይስ እንኳን ደህና መጡ። ዛሬ አዲስ ታሪክ እናቀርባለን።'
print('Generating Amharic with authorized reference...')
try:
 result=model.generate(text=text,ref_audio=ref,ref_text=None)
except Exception as e:
 print('Generation failed:',repr(e));raise
wave=result[0]
if isinstance(wave,torch.Tensor):wave=wave.detach().cpu().float().numpy()
wave=np.asarray(wave,dtype=np.float32).squeeze()
assert wave.ndim==1 and len(wave)>0,'Unexpected audio shape'
out='/content/zemavoice_stage11b_female_test.wav'
sf.write(out,wave,24000)
display(Audio(wave,rate=24000))
print('Saved',out)


In [ ]:
files.download('/content/zemavoice_stage11b_female_test.wav')
